# Phase 2 driver

A **driver only** (CLAUDE2.md "Code layout"). Every cell here imports `src/`
code to launch a run or look at a result. No loader, training loop, config
system or baseline logic is defined in this notebook — that lives in `src/`
with tests and is run through `python -m scripts.t20x_*`.

Run `jupyter lab` from the repo root. Strip outputs before committing:
`nbstripout notebooks/phase2.ipynb`.

In [1]:
import os
import pathlib
import sys

# Work from the repo root so `import src...` and the anchored corpus paths resolve
# whether the kernel started here or in notebooks/.
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

In [2]:
# T-201 — environment guard. Same check every entry point runs.
from src.env_check import check_env

check_env()

/home/souparno/Projects/Cross-Linguistic_Transfer_Diagnotic_for_Financial_NLP_in_Indic_Languages/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.11.15',
 'executable': '/home/souparno/Projects/Cross-Linguistic_Transfer_Diagnotic_for_Financial_NLP_in_Indic_Languages/env/bin/python',
 'packages': {'torch': '2.14.0+cu130',
  'numpy': '2.4.6',
  'scikit-learn': '1.9.0',
  'PyYAML': '6.0.3',
  'transformers': '4.57.6',
  'pandas': '3.0.5',
  'pyarrow': '25.0.1'},
 'cuda_available': True,
 'device_count': 1,
 'device_name': 'NVIDIA GeForce RTX 3050 Laptop GPU'}

## T-202 — data loading

`src/data.py`. Native splits only for training (`hard rule 1`); `load_split`
asserts the origin so a machine-translated split can't slip in.

In [3]:
from src import data as D

TASK = 2  # 2 = sustainability (binary), 3 = ESG themes (10-class)
frame = D.load_native(TASK, "hin")
print(len(frame), "rows", "| labels:", D.label_names(TASK))
frame["label"].value_counts(normalize=True)

2238 rows | labels: ['sustainable', 'unsustainable']


label
sustainable      0.541555
unsustainable    0.458445
Name: proportion, dtype: float64

In [4]:
# Seeded, class-stratified train/dev/test. Same seed -> identical partition.
parts = D.stratified_split(frame, seed=0, dev=0.1, test=0.1)
{name: len(part) for name, part in parts.items()}

{'train': 1790, 'dev': 224, 'test': 224}

In [5]:
# Tokenise a split and look at one padded batch. mBERT here because it needs no
# gated-repo access; swap for 'indicbert-v2' once its terms are accepted.
ds = D.load_dataset(TASK, "hin", "mbert-base", max_len=128)
batch = ds.collate([ds[i] for i in range(8)])
{k: (v.shape if hasattr(v, "shape") else v) for k, v in batch.items()}

{'input_ids': torch.Size([8, 128]),
 'attention_mask': torch.Size([8, 128]),
 'labels': torch.Size([8]),
 'item_ids': ['t2_8f3e9d9805a0',
  't2_H_8c45bbc09229',
  't2_H_eaf327d5cc99',
  't2_edb0fc5f843f',
  't2_534c62b9a738',
  't2_fdacea994d52',
  't2_d2592528d265',
  't2_H_34b063b86238']}

## T-203 — training loop

Built in `src/train.py`; run the correctness gate with
`python -m scripts.t203_smoke --device cuda`. To watch a curve here, call
`src.config.run_training(cfg, device="cuda")` on a small `RunConfig` and plot
`result["history"]`.

## T-205 — VRAM budgets

Measured: `reports/vram_budget.md`. Re-probe with
`python -m scripts.t205_vram --full-epoch`. Both shipped configs train an epoch
under ~1.2 GiB, well inside the 3050.

## T-206 — fine-tune task 2 and task 3 separately

Two independent runs, each fine-tuning IndicBERT-v2 on its own native Hindi
split from a shipped config (`configs/train/task{2,3}_hin_indicbert.yaml`).
This cell *drives* `src/config.run_training` — the loop, checkpointing and
determinism all live in `src/`, not here (CLAUDE2.md "Code layout").

Change `SEED` and re-run a cell to fine-tune another seed; the full 3-seed
sweep with resumability and report-writing is `scripts/t206_baseline.py`.

In [6]:
from pathlib import Path

from src.checkpoints import checkpoint_dir, checkpoint_path
from src.config import load_run_config, run_training
from src.experiments import log_run

SEED = 0  # change and re-run to fine-tune another seed
CONFIG = "configs/train/task2_hin_indicbert.yaml"

run = load_run_config(CONFIG, seed=SEED)
run_id = f"{Path(CONFIG).stem}_seed{SEED}"

metrics = run_training(run, device="cuda", work_dir=checkpoint_dir(run_id), progress=print)
log_run(run, run_id, metrics)

print(f"\nrun_id: {run_id}")
print(f"macro-F1 (test): {metrics['macro_f1']:.4f}  accuracy: {metrics['accuracy']:.4f}")
print(f"weights: {checkpoint_path(run_id)}")

resumed from epoch 18 (/home/souparno/Projects/Cross-Linguistic_Transfer_Diagnotic_for_Financial_NLP_in_Indic_Languages/checkpoints/task2_hin_indicbert_seed0/checkpoint.pt)
epoch 18: loss 0.0060 train_f1 0.9989 dev_f1 0.8835
early stop: no dev improvement for 5 epochs

run_id: task2_hin_indicbert_seed0
macro-F1 (test): 0.8194  accuracy: 0.8214
weights: /home/souparno/Projects/Cross-Linguistic_Transfer_Diagnotic_for_Financial_NLP_in_Indic_Languages/checkpoints/task2_hin_indicbert_seed0/checkpoint.pt


In [7]:
from pathlib import Path

from src.checkpoints import checkpoint_dir, checkpoint_path
from src.config import load_run_config, run_training
from src.experiments import log_run

SEED = 0
CONFIG = "configs/train/task3_hin_indicbert.yaml"

run = load_run_config(CONFIG, seed=SEED)
run_id = f"{Path(CONFIG).stem}_seed{SEED}"

metrics = run_training(run, device="cuda", work_dir=checkpoint_dir(run_id), progress=print)
log_run(run, run_id, metrics)

print(f"\nrun_id: {run_id}")
print(f"macro-F1 (test): {metrics['macro_f1']:.4f}  accuracy: {metrics['accuracy']:.4f}")
print(f"weights: {checkpoint_path(run_id)}")

resumed from epoch 40 (/home/souparno/Projects/Cross-Linguistic_Transfer_Diagnotic_for_Financial_NLP_in_Indic_Languages/checkpoints/task3_hin_indicbert_seed0/checkpoint.pt)

run_id: task3_hin_indicbert_seed0
macro-F1 (test): 0.1620  accuracy: 0.2375
weights: /home/souparno/Projects/Cross-Linguistic_Transfer_Diagnotic_for_Financial_NLP_in_Indic_Languages/checkpoints/task3_hin_indicbert_seed0/checkpoint.pt


### Where the fine-tuned weights land

Each run writes to `checkpoints/<run_id>/`, where `run_id` is
`<config-name>_seed<N>` (e.g. `task2_hin_indicbert_seed0`):

- **`checkpoint.pt`** — the full training state, rewritten after every epoch:
  model weights, optimiser/scheduler state, RNG state, and a snapshot of the
  **best** (highest dev macro-F1) epoch's weights under `blob["best"]["state"]`.
  `run_training` already reloads that best snapshot into the model before
  returning, so the metrics printed above are for the best epoch, not the last.
- **`run_config.json`** — the resolved config + its hash, for reference.

`checkpoints/` is **gitignored** — weights are large (~0.5 GB each) and
regenerable from a config + seed. `experiments.csv` and `reports/` are the
durable, committed record of what each run scored.

Retrieve any run later by its `run_id` alone (T-208):

In [8]:
from src import checkpoints as C

for run_id in ("task2_hin_indicbert_seed0", "task3_hin_indicbert_seed0"):
    meta = C.metadata(run_id)
    print(run_id, "->", meta)

# weights = C.best_state_dict("task2_hin_indicbert_seed0")  # a state_dict to reload

task2_hin_indicbert_seed0 -> {'run_id': 'task2_hin_indicbert_seed0', 'config_hash': '900790d8392986d7', 'run_hash': '241c6ddff33229dd', 'last_epoch': 18, 'epochs_recorded': 19, 'best_epoch': 12, 'best_dev_macro_f1': 0.8965773361437318, 'has_best_weights': True, 'size_bytes': 530456111, 'mtime': 1789103000.1878273}
task3_hin_indicbert_seed0 -> {'run_id': 'task3_hin_indicbert_seed0', 'config_hash': '489499b57f506ac2', 'run_hash': None, 'last_epoch': 39, 'epochs_recorded': 40, 'best_epoch': 34, 'best_dev_macro_f1': 0.18922018549378042, 'has_best_weights': True, 'size_bytes': 530555759, 'mtime': 1789074899.02142}


### Compare every logged run

Reads back everything logged so far — from these cells or from
`scripts.t206_baseline`.

In [9]:
import pandas as pd
from src.experiments import read_results

exp = pd.DataFrame(read_results())
display(exp)

try:
    baselines = pd.read_parquet("reports/baselines.parquet")
    summary = (baselines.groupby("config")["macro_f1"]
               .agg(["mean", "std", "count"]).round(4))
    display(summary)
except FileNotFoundError:
    print("no reports/baselines.parquet yet — run scripts.t206_baseline")

,run_id,encoder,split,seed,config_hash,accuracy,macro_f1,date
0,task2_hin_indicbert_seed0,indicbert-v2,task2/H/hin/native,0,ad7b3e2d6188ef3c,0.803571,0.80343,2026-09-10T20:16:30+00:00
1,task2_hin_indicbert_seed1,indicbert-v2,task2/H/hin/native,1,0e23c0d9c1237a67,0.852679,0.851109,2026-09-10T20:20:52+00:00
2,task2_hin_indicbert_seed2,indicbert-v2,task2/H/hin/native,2,5eecc1718a6b0edc,0.821429,0.821414,2026-09-10T20:23:49+00:00
3,task3_hin_indicbert_seed0,indicbert-v2,task3/H/hin/native,0,5a9b91d2f1b0bbd0,0.3125,0.172316,2026-09-10T20:24:42+00:00
4,task3_hin_indicbert_seed1,indicbert-v2,task3/H/hin/native,1,439f8ef75782c29a,0.225,0.096863,2026-09-10T20:25:32+00:00
5,task3_hin_indicbert_seed2,indicbert-v2,task3/H/hin/native,2,3cfa977e0796d337,0.25,0.111562,2026-09-10T20:26:21+00:00
6,task3_hin_indicbert_seed0,indicbert-v2,task3/H/hin/native,0,3c26eb850cc022ce,0.2375,0.162002,2026-09-10T21:14:59+00:00
7,task3_hin_indicbert_seed1,indicbert-v2,task3/H/hin/native,1,0ff17fac80ff5047,0.25,0.171425,2026-09-10T21:16:22+00:00
8,task3_hin_indicbert_seed2,indicbert-v2,task3/H/hin/native,2,679af8829a83a21a,0.1875,0.124289,2026-09-10T21:18:38+00:00
9,task2_hin_indicbert_seed0,indicbert-v2,task2/H/hin/native,0,241c6ddff33229dd,0.821429,0.819355,2026-09-10T21:39:40+00:00


,mean,std,count
config,,,
task2_hin_indicbert.yaml,0.8226,0.0044,3
task3_hin_indicbert.yaml,0.1526,0.0249,3


## T-207 — baseline gap

Load `experiments.csv`, compare the Hindi macro-F1 to IndicFinNLP's published
number, inspect misclassified rows to diagnose any gap before touching
hyperparameters.